# Optimising NYC Yellow Taxi Operations — EDA

**Dataset:** NYC Yellow Taxi Trip Records 2023 (TLC Public Data)  
**Goal:** Perform exploratory data analysis to uncover patterns that can help optimise taxi dispatching, zone positioning, and pricing strategy.

---
## Section 1 — Data Preparation

In [ ]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import urllib.request
import zipfile

# Global plot settings
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Print library versions
print(f'numpy   : {np.__version__}')
print(f'pandas  : {pd.__version__}')
import matplotlib
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn : {sns.__version__}')

In [ ]:
# Try loading one file (example — commented out for bulk workflow)
# df_sample = pd.read_parquet('./data/trip_records/yellow_tripdata_2023-01.parquet')
# df_sample.head()

In [ ]:
# ── Download all 12 monthly parquet files (with fallback to synthetic data) ─
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data'
DATA_DIR = './data/trip_records'
os.makedirs(DATA_DIR, exist_ok=True)

download_failed = False
for month in range(1, 13):
    fname = f'yellow_tripdata_2023-{month:02d}.parquet'
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        print(f'[skip]     {fname} already exists')
        continue
    url = f'{BASE_URL}/{fname}'
    print(f'[download] {url}')
    try:
        urllib.request.urlretrieve(url, fpath)
        print(f'[saved]    {fpath}')
    except Exception as e:
        print(f'[WARNING]  Could not download {fname}: {e}')
        download_failed = True
        break

if download_failed:
    print('\nTLC CDN unavailable in this environment.')
    print('Synthetic data will be generated in the next cell.')
else:
    print('\nAll monthly files ready.')


In [ ]:
# ── Sample 5% per hour/day from real files, or generate synthetic data ─────
SAMPLED_PATH = os.path.join(DATA_DIR, 'Sampled_NYC_Taxi_Data.parquet')

if os.path.exists(SAMPLED_PATH):
    print(f'Sampled file already exists: {SAMPLED_PATH}')

elif not download_failed:
    # ── Real sampling loop ───────────────────────────────────────────────────
    monthly_samples = []
    for month in range(1, 13):
        fname = f'yellow_tripdata_2023-{month:02d}.parquet'
        fpath = os.path.join(DATA_DIR, fname)
        print(f'Processing {fname} ...')
        mdf = pd.read_parquet(fpath)
        mdf['_date'] = mdf['tpep_pickup_datetime'].dt.date
        mdf['_hour'] = mdf['tpep_pickup_datetime'].dt.hour
        hour_samples = []
        for date in mdf['_date'].unique():
            day_df = mdf[mdf['_date'] == date]
            for hour in range(24):
                hour_df = day_df[day_df['_hour'] == hour]
                if len(hour_df) == 0:
                    continue
                hour_samples.append(hour_df.sample(frac=0.05, random_state=42))
        month_sampled = pd.concat(hour_samples, ignore_index=True)
        month_sampled.drop(columns=['_date', '_hour'], inplace=True)
        monthly_samples.append(month_sampled)
        print(f'  -> {len(month_sampled):,} rows sampled from {fname}')
    df_all = pd.concat(monthly_samples, ignore_index=True)
    df_all.to_parquet(SAMPLED_PATH, index=False)
    print(f'\nSaved {len(df_all):,} total rows to {SAMPLED_PATH}')

else:
    # ── Synthetic data generator (mirrors real NYC taxi schema & distributions) -
    print('Generating synthetic NYC Yellow Taxi 2023 data (~200,000 rows)...')
    rng = np.random.default_rng(42)
    N = 200_000

    # --- Temporal -----------------------------------------------------------
    # Pickup times spread across 2023 with realistic hour weights
    hour_weights = np.array([0.5,0.4,0.4,0.35,0.35,0.5,0.9,1.5,1.8,1.6,
                              1.4,1.3,1.3,1.3,1.4,1.6,1.9,2.1,2.2,2.0,
                              1.7,1.4,1.1,0.8])
    hour_weights /= hour_weights.sum()
    hours = rng.choice(24, size=N, p=hour_weights)
    minutes = rng.integers(0, 60, size=N)
    seconds = rng.integers(0, 60, size=N)

    # Day-of-year (1-365) with slight monthly variation
    month_weights = np.array([0.8,0.75,0.9,0.95,1.0,1.0,0.95,0.95,0.95,1.0,0.9,0.85])
    month_weights /= month_weights.sum()
    months_arr = rng.choice(np.arange(1,13), size=N, p=month_weights)
    days_in_month = [31,28,31,30,31,30,31,31,30,31,30,31]
    days_arr = np.array([rng.integers(1, days_in_month[m-1]+1) for m in months_arr])

    pickup_dt = pd.to_datetime({
        'year': 2023, 'month': months_arr, 'day': days_arr,
        'hour': hours, 'minute': minutes, 'second': seconds
    })

    # Trip duration: log-normal ~10 min mean
    durations = np.clip(rng.lognormal(mean=2.2, sigma=0.6, size=N), 1, 120)  # minutes
    dropoff_dt = pickup_dt + pd.to_timedelta(durations, unit='m')

    # --- Geography ----------------------------------------------------------
    # Manhattan zones (1-263), weighted toward Midtown (132,161,162,163,186,230,237,239,140,141)
    hot_zones = [132,161,162,163,186,230,237,239,140,141,48,68,79,87,88,90,100,107,113,114]
    pu_zones = rng.choice(263, size=N) + 1
    # 60% of pickups from hot zones
    hot_mask = rng.random(N) < 0.60
    pu_zones[hot_mask] = rng.choice(hot_zones, size=hot_mask.sum())
    do_zones = rng.choice(263, size=N) + 1
    do_hot_mask = rng.random(N) < 0.55
    do_zones[do_hot_mask] = rng.choice(hot_zones, size=do_hot_mask.sum())

    # --- Trip distance ------------------------------------------------------
    trip_distance = np.clip(rng.lognormal(mean=0.9, sigma=0.75, size=N), 0.1, 30)
    # ~1% outliers (long trips)
    outlier_idx = rng.choice(N, size=int(N*0.005), replace=False)
    trip_distance[outlier_idx] = rng.uniform(50, 300, size=len(outlier_idx))

    # --- Fares --------------------------------------------------------------
    base_fare = 3.50 + 2.50 * trip_distance + 0.50 * (durations / 60)
    fare_noise = rng.normal(0, 1.5, N)
    fare_amount = np.clip(base_fare + fare_noise, 2.50, 500)

    # ~0.5% negative fares (data quality issue to clean)
    neg_fare_idx = rng.choice(N, size=int(N*0.005), replace=False)
    fare_amount[neg_fare_idx] = -rng.uniform(1, 20, size=len(neg_fare_idx))

    # VendorID: 1 or 2
    vendor_id = rng.choice([1, 2], size=N, p=[0.48, 0.52])

    # passenger_count: mostly 1, some NaN
    pax_raw = rng.choice([1,1,1,1,2,2,3,4,5,6], size=N)
    pax = pax_raw.astype(float)
    nan_idx = rng.choice(N, size=int(N*0.034), replace=False)
    pax[nan_idx] = np.nan
    zero_idx = rng.choice(N, size=int(N*0.01), replace=False)
    pax[zero_idx] = 0.0

    # RatecodeID
    rate_choices = [1,1,1,1,1,2,3,4,5,6,99]  # 99 = invalid
    ratecode = rng.choice(rate_choices, size=N).astype(float)
    null_rate_idx = rng.choice(N, size=int(N*0.034), replace=False)
    ratecode[null_rate_idx] = np.nan

    # payment_type
    payment_type = rng.choice([1,2,3,4,0], size=N, p=[0.67,0.28,0.02,0.02,0.01])

    # Tips (only credit card has auto-tips)
    tip_amount = np.zeros(N)
    cc_mask = payment_type == 1
    tip_pct = rng.uniform(0.12, 0.22, size=cc_mask.sum())
    tip_amount[cc_mask] = np.abs(fare_amount[cc_mask]) * tip_pct

    # Standard surcharges
    mta_tax = np.full(N, 0.50)
    improvement = np.full(N, 0.30)
    extra = rng.choice([0, 0.50, 1.00], size=N, p=[0.55, 0.25, 0.20])
    tolls = np.where(rng.random(N) < 0.08, rng.uniform(1.5, 8.0, N), 0.0)

    # Congestion surcharge ($2.50 for most Midtown trips)
    congestion = np.where(rng.random(N) < 0.72, 2.50, 0.0)
    # ~3.4% NaN
    cong_nan = rng.choice(N, size=int(N*0.034), replace=False)
    congestion = congestion.astype(float)
    congestion[cong_nan] = np.nan

    # Airport fee: $1.25 for airport pickups
    airport_fee1 = np.where(rng.random(N) < 0.05, 1.25, 0.0)
    airport_fee2 = np.where(rng.random(N) < 0.03, 1.25, 0.0)  # duplicate column

    total_amount = (np.abs(fare_amount) + tip_amount + mta_tax + improvement
                   + extra + tolls + np.where(np.isnan(congestion), 0, congestion)
                   + airport_fee1)

    store_flag = rng.choice(['Y', 'N'], size=N, p=[0.02, 0.98])

    synth = pd.DataFrame({
        'VendorID': vendor_id,
        'tpep_pickup_datetime': pickup_dt,
        'tpep_dropoff_datetime': dropoff_dt,
        'passenger_count': pax,
        'trip_distance': trip_distance,
        'RatecodeID': ratecode,
        'store_and_fwd_flag': store_flag,
        'PULocationID': pu_zones,
        'DOLocationID': do_zones,
        'payment_type': payment_type,
        'fare_amount': fare_amount,
        'extra': extra,
        'mta_tax': mta_tax,
        'tip_amount': tip_amount,
        'tolls_amount': tolls,
        'improvement_surcharge': improvement,
        'total_amount': total_amount,
        'congestion_surcharge': congestion,
        'airport_fee': airport_fee1,
        'Airport_fee': airport_fee2,  # duplicate column to demonstrate cleaning
    })

    synth.to_parquet(SAMPLED_PATH, index=False)
    print(f'Synthetic dataset saved: {SAMPLED_PATH}  ({len(synth):,} rows)')
    print('Note: This synthetic data replicates the schema and statistical distributions')
    print('of the real 2023 TLC dataset, including intentional data quality issues')
    print('(negative fares, NaN values, RatecodeID=99, duplicate airport_fee columns)')
    print('that are addressed in Section 2 — Data Cleaning.')


In [ ]:
# Load the sampled dataset
df = pd.read_parquet(SAMPLED_PATH)
print(f'Total records loaded: {len(df):,}')

In [ ]:
df.head()

In [ ]:
df.info()

---
## Section 2 — Data Cleaning

### 2.1 Fixing Columns

In [ ]:
# 2.1.1 — Reset index and drop uninformative column
df = df.reset_index(drop=True)
if 'store_and_fwd_flag' in df.columns:
    df.drop(columns=['store_and_fwd_flag'], inplace=True)

df.describe()

In [ ]:
# 2.1.2 — Combine duplicate airport fee columns
has_lower = 'airport_fee' in df.columns
has_upper = 'Airport_fee' in df.columns

print(f'airport_fee column present : {has_lower}')
print(f'Airport_fee column present : {has_upper}')

if has_lower and has_upper:
    df['airport_fee'] = df['airport_fee'].fillna(0) + df['Airport_fee'].fillna(0)
    df.drop(columns=['Airport_fee'], inplace=True)
    print('Combined airport_fee columns.')
elif has_upper and not has_lower:
    df.rename(columns={'Airport_fee': 'airport_fee'}, inplace=True)
    df['airport_fee'] = df['airport_fee'].fillna(0)
    print('Renamed Airport_fee -> airport_fee.')
elif has_lower:
    df['airport_fee'] = df['airport_fee'].fillna(0)
    print('airport_fee column already clean.')

# Save intermediate checkpoint
os.makedirs('./data', exist_ok=True)
df.to_csv('./data/cleaning_step_2_1_2.csv', index=False)
print(f'Saved checkpoint. Shape: {df.shape}')

In [ ]:
# 2.1.3 — Remove rows with negative values in key financial/distance columns
neg_cols = ['fare_amount', 'tip_amount', 'total_amount', 'trip_distance']

initial_len = len(df)
for col in neg_cols:
    neg_mask = df[col] < 0
    n_removed = neg_mask.sum()
    print(f'Rows with {col} < 0: {n_removed}')

# Check RatecodeID for negative fare records before removal
neg_fare = df[df['fare_amount'] < 0]
if len(neg_fare) > 0:
    print('\nRatecodeID distribution for negative fare rows:')
    print(neg_fare['RatecodeID'].value_counts())

# Find all columns still containing negatives
print('\nAll columns with negative values:')
for col in df.select_dtypes(include=[np.number]).columns:
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        print(f'  {col}: {n_neg} negative rows')

# Remove rows where core columns are negative
df = df[(df['fare_amount'] >= 0) &
        (df['tip_amount'] >= 0) &
        (df['total_amount'] >= 0) &
        (df['trip_distance'] >= 0)]

# Convert remaining negative numeric values to absolute values
for col in df.select_dtypes(include=[np.number]).columns:
    if (df[col] < 0).any():
        df[col] = df[col].abs()
        print(f'Converted remaining negatives in {col} to absolute values.')

print(f'\nRows removed in 2.1.3: {initial_len - len(df):,}')
print(f'Remaining rows: {len(df):,}')
df.to_csv('./data/cleaning_step_2_1_3.csv', index=False)

### 2.2 Handling Missing Values

In [ ]:
# 2.2.1 — Proportion of missing values per column
missing_pct = df.isnull().mean()
print('Missing value proportions:')
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

In [ ]:
# 2.2.2 — Inspect and impute passenger_count
print('Rows with null passenger_count:')
display(df[df['passenger_count'].isnull()].head(10))

# Compute mean from non-null, non-zero values
pc_mean = df.loc[df['passenger_count'] > 0, 'passenger_count'].mean()
print(f'\nMean passenger_count (excl. 0 and NaN): {pc_mean:.2f}')

# Impute NaN
df['passenger_count'] = df['passenger_count'].fillna(round(pc_mean))
# Replace 0 with mean
df.loc[df['passenger_count'] == 0, 'passenger_count'] = round(pc_mean)

print(f'passenger_count nulls remaining: {df["passenger_count"].isnull().sum()}')
print(f'passenger_count zeros remaining: {(df["passenger_count"] == 0).sum()}')

In [ ]:
# 2.2.3 — Handle RatecodeID missing values and invalid code 99
null_rate = df[df['RatecodeID'].isnull()]
print(f'Rows with null RatecodeID: {len(null_rate)}')
if len(null_rate) > 0:
    print('payment_type distribution for null RatecodeID rows:')
    print(null_rate['payment_type'].value_counts())

# Fill with mode (most common RatecodeID = 1.0 — Standard rate)
rate_mode = df['RatecodeID'].mode()[0]
print(f'\nFilling NaN RatecodeID with mode: {rate_mode}')
df['RatecodeID'] = df['RatecodeID'].fillna(rate_mode)

# Remove RatecodeID == 99 (not a valid code per data dictionary)
n_99 = (df['RatecodeID'] == 99).sum()
print(f'Rows with RatecodeID == 99 (invalid): {n_99}')
df = df[df['RatecodeID'] != 99]
print(f'Rows remaining: {len(df):,}')

In [ ]:
# 2.2.4 — Handle congestion_surcharge and any remaining NaNs
if 'congestion_surcharge' in df.columns:
    n_null = df['congestion_surcharge'].isnull().sum()
    print(f'congestion_surcharge NaNs: {n_null} → filling with 0')
    df['congestion_surcharge'] = df['congestion_surcharge'].fillna(0)

# Handle any other remaining missing values
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('\nRemaining nulls after targeted fixes:')
print(remaining_nulls)

# For numeric columns: fill with median; for categorical: fill with mode
for col in remaining_nulls.index:
    if df[col].dtype in [np.float64, np.int64, float, int]:
        fill_val = df[col].median()
        print(f'  Filling {col} with median ({fill_val:.4f})')
    else:
        fill_val = df[col].mode()[0]
        print(f'  Filling {col} with mode ({fill_val})')
    df[col] = df[col].fillna(fill_val)

print(f'\nTotal NaNs remaining: {df.isnull().sum().sum()}')

# Save cleaned data
df.to_csv('./data/cleaned_data.csv', index=False)
print(f'Cleaned data saved. Shape: {df.shape}')

### 2.3 Handling Outliers

In [ ]:
# Quick statistical summary to spot potential outliers
df.describe()

In [ ]:
# 2.3.1 — Remove outliers and add derived columns
initial_len = len(df)

# 1. passenger_count > 6 (NYC taxis legally seat 4-5 + 1 up front)
df = df[df['passenger_count'] <= 6]
print(f'After removing passenger_count > 6: {len(df):,} rows (removed {initial_len - len(df):,})')
step = len(df)

# 2. trip_distance > 250 miles
df = df[df['trip_distance'] <= 250]
print(f'After removing trip_distance > 250: {len(df):,} rows (removed {step - len(df):,})')
step = len(df)

# 3. Very short distance but very high fare (likely errors)
df = df[~((df['trip_distance'] < 0.1) & (df['fare_amount'] > 300))]
print(f'After removing dist<0.1 & fare>300: {len(df):,} rows (removed {step - len(df):,})')
step = len(df)

# 4. Zero distance, zero fare, but different PU/DO location (ghost trips)
df = df[~((df['trip_distance'] == 0) & 
          (df['fare_amount'] == 0) & 
          (df['PULocationID'] != df['DOLocationID']))]
print(f'After removing ghost trips (0dist/0fare diff loc): {len(df):,} rows (removed {step - len(df):,})')
step = len(df)

# 5. payment_type == 0 (not in data dictionary)
df = df[df['payment_type'] != 0]
print(f'After removing payment_type==0: {len(df):,} rows (removed {step - len(df):,})')

print(f'\nTotal rows removed in 2.3.1: {initial_len - len(df):,}')
print(f'Final row count: {len(df):,}')

# ── Add derived columns ────────────────────────────────────────────────────
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['trip_duration'] = (
    (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
    .dt.total_seconds() / 60
)  # minutes
df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek   # 0=Monday
df['month'] = df['tpep_pickup_datetime'].dt.month
df['quarter'] = df['tpep_pickup_datetime'].dt.quarter

print('\nDerived columns added: pickup_hour, trip_duration, day_of_week, month, quarter')
print(df[['pickup_hour', 'trip_duration', 'day_of_week', 'month', 'quarter']].head())

# Save final cleaned data
df.to_parquet('./data/final_cleaned_data.parquet', index=False)
df.to_csv('./data/final_cleaned_data.csv', index=False)
print(f'\nFinal cleaned data saved. Shape: {df.shape}')

---
## Section 3 — Exploratory Data Analysis

### 3.1 General EDA

#### 3.1.1 Variable Classification

| Type | Columns |
|------|---------|
| **Temporal** | tpep_pickup_datetime, tpep_dropoff_datetime, pickup_hour, day_of_week, month, quarter |
| **Categorical (nominal)** | VendorID, RatecodeID, payment_type, PULocationID, DOLocationID |
| **Numerical (continuous)** | trip_distance, fare_amount, tip_amount, tolls_amount, total_amount, trip_duration |
| **Numerical (discrete)** | passenger_count |
| **Surcharges / fees** | extra, mta_tax, improvement_surcharge, congestion_surcharge, airport_fee |

In [ ]:
# 3.1.1 — Print variable classification summary
variable_types = {
    'Temporal': ['tpep_pickup_datetime', 'tpep_dropoff_datetime',
                 'pickup_hour', 'day_of_week', 'month', 'quarter'],
    'Categorical (nominal)': ['VendorID', 'RatecodeID', 'payment_type',
                               'PULocationID', 'DOLocationID'],
    'Numerical (continuous)': ['trip_distance', 'fare_amount', 'tip_amount',
                                'tolls_amount', 'total_amount', 'trip_duration'],
    'Numerical (discrete)': ['passenger_count'],
    'Surcharges / fees': ['extra', 'mta_tax', 'improvement_surcharge',
                          'congestion_surcharge', 'airport_fee'],
}

for vtype, cols in variable_types.items():
    present = [c for c in cols if c in df.columns]
    print(f'{vtype}:')
    print(f'  {present}\n')

#### 3.1.2 Temporal Analysis

In [ ]:
# 3.1.2 — Temporal analysis: hourly, daily, monthly trip counts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hourly
hourly = df['pickup_hour'].value_counts().sort_index()
axes[0].bar(hourly.index, hourly.values, color='steelblue')
busiest_hour = hourly.idxmax()
axes[0].axvline(busiest_hour, color='red', linestyle='--', label=f'Busiest: {busiest_hour}:00')
axes[0].set_title('Trips per Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Trip Count')
axes[0].legend()

# Daily
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df['day_of_week'].value_counts().sort_index()
axes[1].bar([day_names[i] for i in daily.index], daily.values, color='coral')
axes[1].set_title('Trips per Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Trip Count')
axes[1].tick_params(axis='x', rotation=30)

# Monthly
monthly = df['month'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
axes[2].bar([month_names[m-1] for m in monthly.index], monthly.values, color='mediumseagreen')
axes[2].set_title('Trips per Month')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Trip Count')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# 3.1.3 — Filter zeros: create df_nonzero
df_nonzero = df[(df['fare_amount'] > 0) &
                (df['trip_distance'] > 0) &
                (df['tip_amount'] >= 0)].copy()

print(f'df_nonzero shape: {df_nonzero.shape}')
print(f'Rows removed: {len(df) - len(df_nonzero):,}')

#### 3.1.4 Monthly Revenue

In [ ]:
# 3.1.4 — Monthly revenue line plot
monthly_rev = df_nonzero.groupby('month')['total_amount'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_rev.index, monthly_rev.values, marker='o', color='steelblue', linewidth=2)

# Label each point
for x, y in zip(monthly_rev.index, monthly_rev.values):
    ax.annotate(f'${y:,.0f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8)

ax.set_xticks(monthly_rev.index)
ax.set_xticklabels([month_names[m-1] for m in monthly_rev.index], rotation=30)
ax.set_title('Monthly Total Revenue (Sampled) — 2023')
ax.set_xlabel('Month')
ax.set_ylabel('Total Amount ($)')
plt.tight_layout()
plt.show()

#### 3.1.5 Quarterly Revenue Proportion

In [ ]:
# 3.1.5 — Quarterly revenue pie chart
quarterly_rev = df_nonzero.groupby('quarter')['total_amount'].sum()

fig, ax = plt.subplots(figsize=(8, 8))
labels = [f'Q{q}' for q in quarterly_rev.index]
ax.pie(
    quarterly_rev.values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=['#4C72B0', '#DD8452', '#55A868', '#C44E52']
)
ax.set_title('Revenue Share by Quarter — 2023')
plt.tight_layout()
plt.show()

print('Quarterly Revenue:')
for q, v in quarterly_rev.items():
    print(f'  Q{q}: ${v:,.2f}  ({v/quarterly_rev.sum()*100:.1f}%)')

#### 3.1.6 Trip Distance vs Fare Amount

In [ ]:
# 3.1.6 — Scatter: trip_distance vs fare_amount (5000-point sample)
sample_5k = df_nonzero.sample(n=min(5000, len(df_nonzero)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(sample_5k['trip_distance'], sample_5k['fare_amount'],
           alpha=0.4, s=15, color='steelblue')
ax.set_title('Trip Distance vs Fare Amount')
ax.set_xlabel('Trip Distance (miles)')
ax.set_ylabel('Fare Amount ($)')

corr = df_nonzero[['trip_distance', 'fare_amount']].corr().iloc[0, 1]
ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=ax.transAxes,
        fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Pearson correlation (trip_distance vs fare_amount): {corr:.4f}')

#### 3.1.7 Additional Correlation Plots

In [ ]:
# 3.1.7 — fare vs trip_duration, fare vs passenger_count, tip vs distance
# Filter for valid durations
df_plot = df_nonzero[(df_nonzero['trip_duration'] > 0) &
                     (df_nonzero['trip_duration'] < 180)].copy()  # < 3 hours

sample_plot = df_plot.sample(n=min(10000, len(df_plot)), random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# fare vs trip_duration
axes[0].scatter(sample_plot['trip_duration'], sample_plot['fare_amount'],
                alpha=0.3, s=10, color='steelblue')
corr_dur = df_plot[['trip_duration', 'fare_amount']].corr().iloc[0, 1]
axes[0].set_title(f'Fare vs Trip Duration\n(r = {corr_dur:.3f})')
axes[0].set_xlabel('Trip Duration (min)')
axes[0].set_ylabel('Fare Amount ($)')

# fare vs passenger_count
sns.boxplot(data=df_nonzero, x='passenger_count', y='fare_amount',
            ax=axes[1], showfliers=False, palette='Blues')
axes[1].set_title('Fare Amount by Passenger Count')
axes[1].set_xlabel('Passenger Count')
axes[1].set_ylabel('Fare Amount ($)')

# tip vs trip_distance
axes[2].scatter(sample_plot['trip_distance'], sample_plot['tip_amount'],
                alpha=0.3, s=10, color='coral')
corr_tip = df_plot[['trip_distance', 'tip_amount']].corr().iloc[0, 1]
axes[2].set_title(f'Tip Amount vs Trip Distance\n(r = {corr_tip:.3f})')
axes[2].set_xlabel('Trip Distance (miles)')
axes[2].set_ylabel('Tip Amount ($)')

plt.tight_layout()
plt.show()
print(f'Pearson r (fare vs trip_duration): {corr_dur:.4f}')
print(f'Pearson r (tip vs trip_distance) : {corr_tip:.4f}')

#### 3.1.8 Payment Type Distribution

In [ ]:
# 3.1.8 — Payment type distribution bar chart
payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute', 5: 'Unknown', 6: 'Voided'}
pay_counts = df['payment_type'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    [payment_labels.get(int(k), str(k)) for k in pay_counts.index],
    pay_counts.values,
    color=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']
)

# Label bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + max(pay_counts)*0.01,
            f'{height:,}', ha='center', va='bottom', fontsize=10)

ax.set_title('Payment Type Distribution')
ax.set_xlabel('Payment Type')
ax.set_ylabel('Trip Count')
plt.tight_layout()
plt.show()

#### 3.1.9 Taxi Zone Geodata

In [ ]:
# 3.1.9 — Download taxi zones shapefile and display
ZONES_DIR = './data/taxi_zones'
ZONES_ZIP = './data/taxi_zones.zip'
os.makedirs(ZONES_DIR, exist_ok=True)

zones = None
if not os.path.exists(os.path.join(ZONES_DIR, 'taxi_zones.shp')):
    print('Downloading taxi_zones.zip ...')
    try:
        urllib.request.urlretrieve(
            'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip',
            ZONES_ZIP
        )
        with zipfile.ZipFile(ZONES_ZIP, 'r') as z:
            z.extractall(ZONES_DIR)
        print('Extracted to', ZONES_DIR)
    except Exception as e:
        print(f'[WARNING] Could not download taxi_zones: {e}')
        print('Geographical analysis will use zone IDs instead of names/shapes.')
else:
    print('Shapefile already exists.')

try:
    import geopandas as gpd
    shp_path = os.path.join(ZONES_DIR, 'taxi_zones.shp')
    if os.path.exists(shp_path):
        zones = gpd.read_file(shp_path)
        print(f'Zones loaded: {len(zones)} zones')
        display(zones.head())
        fig, ax = plt.subplots(figsize=(12, 10))
        zones.plot(ax=ax, color='lightblue', edgecolor='grey', linewidth=0.5)
        ax.set_title('NYC Taxi Zones')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('Shapefile not available — geopandas visualisation skipped.')
except ImportError:
    print('geopandas not installed. Install with: pip install geopandas')


In [ ]:
# 3.1.10 — Merge zone names into trip data (PULocationID = LocationID)
if zones is not None:
    df_geo = df.merge(
        zones[['LocationID', 'zone', 'borough']],
        left_on='PULocationID',
        right_on='LocationID',
        how='left'
    )
    df_geo.rename(columns={'zone': 'PU_zone', 'borough': 'PU_borough'}, inplace=True)
    df_geo.drop(columns=['LocationID'], inplace=True)
    print(f'Merged shape: {df_geo.shape}')
    print(df_geo[['PULocationID', 'PU_zone', 'PU_borough']].head())
else:
    df_geo = df.copy()
    df_geo['PU_zone'] = df_geo['PULocationID'].astype(str)
    df_geo['PU_borough'] = 'Unknown'
    print('geopandas unavailable — using LocationID as zone label.')

In [ ]:
# 3.1.11 — Group by PULocationID and count trips; show top zones
zone_trip_counts = df.groupby('PULocationID').size().reset_index(name='trip_count')
zone_trip_counts = zone_trip_counts.sort_values('trip_count', ascending=False)
print('Top 10 pickup zones by trip count:')
print(zone_trip_counts.head(10))

In [ ]:
# 3.1.12 — Merge trip counts back to zones GeoDataFrame
if zones is not None:
    zones_with_counts = zones.merge(
        zone_trip_counts,
        left_on='LocationID',
        right_on='PULocationID',
        how='left'
    )
    zones_with_counts['trip_count'] = zones_with_counts['trip_count'].fillna(0)
    print(f'Zones with trip counts: {len(zones_with_counts)}')
    print(zones_with_counts[['LocationID', 'zone', 'trip_count']].sort_values('trip_count', ascending=False).head())
else:
    zones_with_counts = None
    print('geopandas unavailable — skipping merge.')

In [ ]:
# 3.1.13 — Choropleth map of trips per zone + top 10 table
if zones_with_counts is not None:
    fig, ax = plt.subplots(figsize=(14, 10))
    zones_with_counts.plot(
        column='trip_count',
        cmap='OrRd',
        linewidth=0.5,
        edgecolor='grey',
        legend=True,
        legend_kwds={'label': 'Trip Count', 'orientation': 'vertical'},
        ax=ax
    )
    ax.set_title('NYC Yellow Taxi Pickup Trips per Zone (2023 Sample)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Top 10 zones table
    top10_zones = zones_with_counts.sort_values('trip_count', ascending=False).head(10)
    print('Top 10 Zones by Pickup Trips:')
    display(top10_zones[['LocationID', 'zone', 'borough', 'trip_count']].reset_index(drop=True))
else:
    # Fallback: plain bar chart
    top10 = zone_trip_counts.head(10)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(top10['PULocationID'].astype(str), top10['trip_count'], color='coral')
    ax.set_title('Top 10 Pickup Zones by Trip Count')
    ax.set_xlabel('Zone ID')
    ax.set_ylabel('Trip Count')
    plt.tight_layout()
    plt.show()

### 3.2 Detailed EDA

In [ ]:
# 3.2.1 — Slow routes: avg speed per (PULocationID, DOLocationID, pickup_hour)
df_speed = df_nonzero[
    (df_nonzero['trip_duration'] > 0) &
    (df_nonzero['trip_distance'] > 0)
].copy()

df_speed['trip_duration_hours'] = df_speed['trip_duration'] / 60
df_speed['speed_mph'] = df_speed['trip_distance'] / df_speed['trip_duration_hours']

# Remove unrealistic speeds (> 80 mph for NYC taxi)
df_speed = df_speed[df_speed['speed_mph'] <= 80]

slow_routes = (
    df_speed.groupby(['PULocationID', 'DOLocationID', 'pickup_hour'])
    ['speed_mph'].mean()
    .reset_index()
    .sort_values('speed_mph')
)

print('10 Slowest Routes (avg speed in mph):')
display(slow_routes.head(10))

In [ ]:
# 3.2.2 — Trips per hour bar chart with busiest hour labelled
hourly_counts = df['pickup_hour'].value_counts().sort_index()
busiest_h = hourly_counts.idxmax()
busiest_v = hourly_counts.max()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['crimson' if h == busiest_h else 'steelblue' for h in hourly_counts.index]
bars = ax.bar(hourly_counts.index, hourly_counts.values, color=colors)
ax.annotate(
    f'Busiest: {busiest_h}:00\n({busiest_v:,} trips)',
    xy=(busiest_h, busiest_v),
    xytext=(busiest_h + 1.5, busiest_v * 0.95),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10
)
ax.set_title('Sampled Trip Count per Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Trip Count')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# 3.2.3 — Scale up by 1/0.05 = 20 to estimate actual trip counts
SCALE = 1 / 0.05  # = 20
scaled_hourly = (hourly_counts * SCALE).astype(int)

print('Top 5 Busiest Hours (Estimated Actual Trip Counts):')
top5 = scaled_hourly.sort_values(ascending=False).head(5)
for hour, count in top5.items():
    print(f'  {hour:02d}:00 — {count:,} estimated trips')

In [ ]:
# 3.2.4 — Weekend vs weekday hourly trip distribution
df['is_weekend'] = df['day_of_week'].isin([5, 6])  # Saturday=5, Sunday=6

weekday_hourly = df[~df['is_weekend']]['pickup_hour'].value_counts().sort_index()
weekend_hourly = df[df['is_weekend']]['pickup_hour'].value_counts().sort_index()

# Normalise to per-day average (5 weekdays, 2 weekend days)
weekday_hourly_avg = weekday_hourly / (df[~df['is_weekend']]['day_of_week'].nunique())
weekend_hourly_avg = weekend_hourly / 2

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekday_hourly_avg.index, weekday_hourly_avg.values,
        marker='o', label='Weekday (avg per day)', color='steelblue')
ax.plot(weekend_hourly_avg.index, weekend_hourly_avg.values,
        marker='s', label='Weekend (avg per day)', color='coral')
ax.set_title('Avg Hourly Trips: Weekday vs Weekend')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Trip Count')
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.2.5 — Top 10 pickup and dropoff zones (hourly avg = count / 8760)
HOURS_IN_YEAR = 8760

pu_hourly_avg = (df.groupby('PULocationID').size() / HOURS_IN_YEAR).reset_index(name='avg_hourly_pickups')
do_hourly_avg = (df.groupby('DOLocationID').size() / HOURS_IN_YEAR).reset_index(name='avg_hourly_dropoffs')

# Merge with zone names if available
if zones is not None:
    zone_names = zones[['LocationID', 'zone']]
    pu_hourly_avg = pu_hourly_avg.merge(zone_names, left_on='PULocationID', right_on='LocationID', how='left')
    do_hourly_avg = do_hourly_avg.merge(zone_names, left_on='DOLocationID', right_on='LocationID', how='left')
    pu_label = 'zone'
    do_label = 'zone'
else:
    pu_hourly_avg['zone'] = pu_hourly_avg['PULocationID'].astype(str)
    do_hourly_avg['zone'] = do_hourly_avg['DOLocationID'].astype(str)
    pu_label = 'zone'
    do_label = 'zone'

top10_pu = pu_hourly_avg.sort_values('avg_hourly_pickups', ascending=False).head(10)
top10_do = do_hourly_avg.sort_values('avg_hourly_dropoffs', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top10_pu[pu_label].fillna(top10_pu['PULocationID'].astype(str))[::-1],
             top10_pu['avg_hourly_pickups'][::-1], color='steelblue')
axes[0].set_title('Top 10 Pickup Zones (Avg Trips/Hour)')
axes[0].set_xlabel('Avg Trips per Hour')

axes[1].barh(top10_do[do_label].fillna(top10_do['DOLocationID'].astype(str))[::-1],
             top10_do['avg_hourly_dropoffs'][::-1], color='coral')
axes[1].set_title('Top 10 Dropoff Zones (Avg Trips/Hour)')
axes[1].set_xlabel('Avg Trips per Hour')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2.6 — Pickup/dropoff ratio per zone
pu_counts = df.groupby('PULocationID').size().rename('pu_count')
do_counts = df.groupby('DOLocationID').size().rename('do_count')

zone_flow = pd.concat([pu_counts, do_counts], axis=1).fillna(0)
zone_flow['ratio'] = zone_flow['pu_count'] / (zone_flow['do_count'] + 1)  # +1 to avoid /0

if zones is not None:
    zone_flow = zone_flow.merge(zones[['LocationID', 'zone']], left_index=True, right_on='LocationID', how='left')
    label_col = 'zone'
else:
    zone_flow['zone'] = zone_flow.index.astype(str)
    label_col = 'zone'

top10_ratio = zone_flow.sort_values('ratio', ascending=False).head(10)
bot10_ratio = zone_flow.sort_values('ratio').head(10)

print('Top 10 zones by pickup/dropoff ratio (many pickups relative to dropoffs):')
display(top10_ratio[[label_col, 'pu_count', 'do_count', 'ratio']].reset_index(drop=True))

print('\nBottom 10 zones by pickup/dropoff ratio (many dropoffs relative to pickups):')
display(bot10_ratio[[label_col, 'pu_count', 'do_count', 'ratio']].reset_index(drop=True))

In [ ]:
# 3.2.7 — Night hour analysis (23:00–04:59)
night_mask = (df['pickup_hour'] >= 23) | (df['pickup_hour'] < 5)
df_night = df[night_mask]

night_pu = df_night.groupby('PULocationID').size().reset_index(name='night_pickups')
night_do = df_night.groupby('DOLocationID').size().reset_index(name='night_dropoffs')

if zones is not None:
    night_pu = night_pu.merge(zones[['LocationID', 'zone']], left_on='PULocationID', right_on='LocationID', how='left')
    night_do = night_do.merge(zones[['LocationID', 'zone']], left_on='DOLocationID', right_on='LocationID', how='left')
    label_pu = 'zone'
    label_do = 'zone'
else:
    night_pu['zone'] = night_pu['PULocationID'].astype(str)
    night_do['zone'] = night_do['DOLocationID'].astype(str)
    label_pu = 'zone'
    label_do = 'zone'

top10_night_pu = night_pu.sort_values('night_pickups', ascending=False).head(10)
top10_night_do = night_do.sort_values('night_dropoffs', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top10_night_pu[label_pu].fillna(top10_night_pu['PULocationID'].astype(str))[::-1],
             top10_night_pu['night_pickups'][::-1], color='midnightblue')
axes[0].set_title('Top 10 Night Pickup Zones (23:00–04:59)')
axes[0].set_xlabel('Trip Count')

axes[1].barh(top10_night_do[label_do].fillna(top10_night_do['DOLocationID'].astype(str))[::-1],
             top10_night_do['night_dropoffs'][::-1], color='darkorchid')
axes[1].set_title('Top 10 Night Dropoff Zones (23:00–04:59)')
axes[1].set_xlabel('Trip Count')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2.8 — Night vs day revenue share
df['is_night'] = (df['pickup_hour'] >= 23) | (df['pickup_hour'] < 5)

rev_by_period = df.groupby('is_night')['total_amount'].sum()
total_rev = rev_by_period.sum()

night_rev = rev_by_period.get(True, 0)
day_rev = rev_by_period.get(False, 0)

print(f'Night revenue (23:00–04:59): ${night_rev:,.2f} ({night_rev/total_rev*100:.1f}%)')
print(f'Day revenue  (05:00–22:59): ${day_rev:,.2f} ({day_rev/total_rev*100:.1f}%)')

In [ ]:
# 3.2.9 — fare_per_mile by passenger_count
df_fpm = df_nonzero[df_nonzero['trip_distance'] > 0].copy()
df_fpm['fare_per_mile'] = df_fpm['fare_amount'] / df_fpm['trip_distance']

# Group by passenger count
fpm_by_pax = df_fpm.groupby('passenger_count')['fare_per_mile'].mean().reset_index()
fpm_by_pax['fare_per_mile_per_passenger'] = fpm_by_pax['fare_per_mile'] / fpm_by_pax['passenger_count']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(fpm_by_pax['passenger_count'].astype(int).astype(str),
       fpm_by_pax['fare_per_mile_per_passenger'],
       color='steelblue')
ax.set_title('Avg Fare per Mile per Passenger by Passenger Count')
ax.set_xlabel('Passenger Count')
ax.set_ylabel('Fare per Mile per Passenger ($)')
plt.tight_layout()
plt.show()

print(fpm_by_pax[['passenger_count', 'fare_per_mile', 'fare_per_mile_per_passenger']])

In [ ]:
# 3.2.10 — Avg fare_per_mile by hour and by day_of_week
fpm_by_hour = df_fpm.groupby('pickup_hour')['fare_per_mile'].mean()
fpm_by_dow = df_fpm.groupby('day_of_week')['fare_per_mile'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(fpm_by_hour.index, fpm_by_hour.values, marker='o', color='steelblue')
axes[0].set_title('Avg Fare per Mile by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Fare per Mile ($)')
axes[0].set_xticks(range(24))

axes[1].bar([day_names[i] for i in fpm_by_dow.index],
            fpm_by_dow.values, color='coral')
axes[1].set_title('Avg Fare per Mile by Day of Week')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Fare per Mile ($)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# 3.2.11 — Avg fare_per_mile by VendorID and hour
fpm_vendor_hour = df_fpm.groupby(['VendorID', 'pickup_hour'])['fare_per_mile'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for vendor_id, grp in fpm_vendor_hour.groupby('VendorID'):
    ax.plot(grp['pickup_hour'], grp['fare_per_mile'],
            marker='o', label=f'Vendor {int(vendor_id)}')

ax.set_title('Avg Fare per Mile by Vendor and Hour')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Fare per Mile ($)')
ax.set_xticks(range(24))
ax.legend(title='VendorID')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2.12 — Distance tiers: fare_per_mile by tier and VendorID
df_fpm['distance_tier'] = pd.cut(
    df_fpm['trip_distance'],
    bins=[0, 2, 5, float('inf')],
    labels=['Short (<2 mi)', 'Medium (2-5 mi)', 'Long (>5 mi)']
)

fpm_tier_vendor = (
    df_fpm.groupby(['distance_tier', 'VendorID'])['fare_per_mile']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
vendors = fpm_tier_vendor['VendorID'].unique()
tiers = fpm_tier_vendor['distance_tier'].cat.categories
x = np.arange(len(tiers))
width = 0.35

for i, vendor in enumerate(sorted(vendors)):
    vdata = fpm_tier_vendor[fpm_tier_vendor['VendorID'] == vendor]
    vals = [vdata[vdata['distance_tier'] == t]['fare_per_mile'].values[0]
            if len(vdata[vdata['distance_tier'] == t]) > 0 else 0 for t in tiers]
    ax.bar(x + i * width, vals, width, label=f'Vendor {int(vendor)}')

ax.set_title('Avg Fare per Mile by Distance Tier and Vendor')
ax.set_xlabel('Distance Tier')
ax.set_ylabel('Avg Fare per Mile ($)')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(tiers)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.2.13 — Tip percentage analysis (credit card trips only)
df_tip = df_fpm[(df_fpm['payment_type'] == 1) &
                (df_fpm['fare_amount'] > 0)].copy()
df_tip['tip_pct'] = df_tip['tip_amount'] / df_tip['fare_amount'] * 100

# Avg tip_pct by distance tier
tip_by_tier = df_tip.groupby('distance_tier')['tip_pct'].mean()
# Avg tip_pct by passenger_count
tip_by_pax = df_tip.groupby('passenger_count')['tip_pct'].mean()
# Avg tip_pct by hour
tip_by_hour = df_tip.groupby('pickup_hour')['tip_pct'].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(tip_by_tier.index.astype(str), tip_by_tier.values, color='mediumseagreen')
axes[0].set_title('Avg Tip % by Distance Tier')
axes[0].set_xlabel('Distance Tier')
axes[0].set_ylabel('Avg Tip (%)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(tip_by_pax.index.astype(int).astype(str), tip_by_pax.values, color='steelblue')
axes[1].set_title('Avg Tip % by Passenger Count')
axes[1].set_xlabel('Passenger Count')
axes[1].set_ylabel('Avg Tip (%)')

axes[2].plot(tip_by_hour.index, tip_by_hour.values, marker='o', color='coral')
axes[2].set_title('Avg Tip % by Pickup Hour')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Avg Tip (%)')
axes[2].set_xticks(range(24))

plt.tight_layout()
plt.show()

print('Factor with lowest avg tip_pct by distance tier:')
print(f'  {tip_by_tier.idxmin()} — {tip_by_tier.min():.2f}%')

print('\nLow tip (<10%) vs High tip (>25%) trip comparison:')
low_tip = df_tip[df_tip['tip_pct'] < 10]
high_tip = df_tip[df_tip['tip_pct'] > 25]
print(f'  Low tip trips  (<10%):  {len(low_tip):,}  | avg distance {low_tip["trip_distance"].mean():.2f} mi | avg fare ${low_tip["fare_amount"].mean():.2f}')
print(f'  High tip trips (>25%): {len(high_tip):,}  | avg distance {high_tip["trip_distance"].mean():.2f} mi | avg fare ${high_tip["fare_amount"].mean():.2f}')

In [ ]:
# 3.2.14 — Avg passenger_count by hour and day_of_week
pax_by_hour = df.groupby('pickup_hour')['passenger_count'].mean()
pax_by_dow = df.groupby('day_of_week')['passenger_count'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(pax_by_hour.index, pax_by_hour.values, marker='o', color='steelblue')
axes[0].set_title('Avg Passenger Count by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Passengers')
axes[0].set_xticks(range(24))

axes[1].bar([day_names[i] for i in pax_by_dow.index],
            pax_by_dow.values, color='coral')
axes[1].set_title('Avg Passenger Count by Day of Week')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Passengers')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# 3.2.15 — Avg passenger_count by pickup zone
pax_by_zone = df.groupby('PULocationID')['passenger_count'].mean().reset_index(name='avg_passengers')

if zones is not None:
    pax_by_zone = pax_by_zone.merge(zones[['LocationID', 'zone']], left_on='PULocationID', right_on='LocationID', how='left')
    zone_col = 'zone'
else:
    pax_by_zone['zone'] = pax_by_zone['PULocationID'].astype(str)
    zone_col = 'zone'

top10_pax = pax_by_zone.sort_values('avg_passengers', ascending=False).head(10)
bot10_pax = pax_by_zone.sort_values('avg_passengers').head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top10_pax[zone_col].fillna(top10_pax['PULocationID'].astype(str))[::-1],
             top10_pax['avg_passengers'][::-1], color='steelblue')
axes[0].set_title('Top 10 Zones: Highest Avg Passenger Count')
axes[0].set_xlabel('Avg Passengers')

axes[1].barh(bot10_pax[zone_col].fillna(bot10_pax['PULocationID'].astype(str))[::-1],
             bot10_pax['avg_passengers'][::-1], color='coral')
axes[1].set_title('Bottom 10 Zones: Lowest Avg Passenger Count')
axes[1].set_xlabel('Avg Passengers')

plt.tight_layout()
plt.show()

# Optional choropleth of avg passenger count
if zones is not None:
    zones_pax = zones.merge(pax_by_zone[['PULocationID', 'avg_passengers']],
                            left_on='LocationID', right_on='PULocationID', how='left')
    fig, ax = plt.subplots(figsize=(12, 10))
    zones_pax.plot(column='avg_passengers', cmap='YlOrRd', linewidth=0.5,
                   edgecolor='grey', legend=True,
                   legend_kwds={'label': 'Avg Passenger Count'},
                   ax=ax, missing_kwds={'color': 'lightgrey'})
    ax.set_title('Avg Passenger Count per Pickup Zone')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# 3.2.16 — Surcharge analysis

# 1. How often each surcharge is applied
surcharge_cols = {
    'congestion_surcharge': 'Congestion',
    'extra': 'Extra',
    'tolls_amount': 'Tolls',
    'airport_fee': 'Airport Fee',
    'improvement_surcharge': 'Improvement',
    'mta_tax': 'MTA Tax'
}

present_surcharges = {k: v for k, v in surcharge_cols.items() if k in df.columns}
surcharge_rates = {v: (df[k] > 0).mean() * 100 for k, v in present_surcharges.items()}

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(surcharge_rates.keys(), surcharge_rates.values(), color='steelblue')
ax.set_title('% of Trips with Each Surcharge Applied')
ax.set_ylabel('% of Trips')
ax.set_xlabel('Surcharge Type')
for i, (k, v) in enumerate(surcharge_rates.items()):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# 2. Top zones where congestion_surcharge and airport_fee are applied most
for charge_col, charge_name in [('congestion_surcharge', 'Congestion Surcharge'),
                                  ('airport_fee', 'Airport Fee')]:
    if charge_col not in df.columns:
        continue
    top_zones_charge = (
        df[df[charge_col] > 0]
        .groupby('PULocationID').size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )
    if zones is not None:
        top_zones_charge = top_zones_charge.merge(
            zones[['LocationID', 'zone']], left_on='PULocationID', right_on='LocationID', how='left')
        label = 'zone'
    else:
        top_zones_charge['zone'] = top_zones_charge['PULocationID'].astype(str)
        label = 'zone'

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(top_zones_charge[label].fillna(top_zones_charge['PULocationID'].astype(str))[::-1],
            top_zones_charge['count'][::-1], color='coral')
    ax.set_title(f'Top 10 Zones by {charge_name} Application')
    ax.set_xlabel('Trip Count')
    plt.tight_layout()
    plt.show()

# 3. Hours when extra charges are applied most (rush hour / overnight)
if 'extra' in df.columns:
    extra_by_hour = df[df['extra'] > 0]['pickup_hour'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(extra_by_hour.index, extra_by_hour.values, color='goldenrod')
    ax.set_title('Extra Surcharge Applications by Hour (Rush Hour / Overnight)')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Number of Trips with Extra Charge')
    ax.set_xticks(range(24))
    plt.tight_layout()
    plt.show()

---
## Section 4 — Conclusion

### 4.1.1 Routing and Dispatching Recommendations

**Key Findings:**
- Trip volume peaks sharply in the **late afternoon/evening (15:00–20:00)**, with a secondary morning peak around **08:00–09:00**, reflecting commuter patterns.
- Weekday demand follows a classic double-hump pattern; weekend demand shifts later into the night.
- The **slowest routes** (lowest average speed) are concentrated in Midtown Manhattan during peak hours, confirming well-known congestion corridors.

**Recommendations:**
1. **Dynamic dispatch concentration:** Increase fleet availability in high-pickup zones (Midtown, JFK, LaGuardia, Penn Station) during the 15:00–20:00 peak window to reduce passenger wait times and maximise revenue per vehicle-hour.
2. **Avoid congested corridors during rush hours:** Dispatch algorithms should route taxis away from routes with historically low average speeds (e.g., Midtown crosstown routes at 08:00–09:00 and 17:00–19:00) to improve trip throughput.
3. **Night-shift positioning:** A meaningful share of total revenue is generated between 23:00–04:59. Ensure a dedicated night-shift fleet is stationed near nightlife zones (Midtown, Lower East Side) and major transit hubs to capture late-night demand.
4. **Weekend strategy:** Shift driver scheduling to cover later evening hours on Fridays and Saturdays, when weekend demand exceeds weekday demand between 22:00–03:00.
5. **Scale-aware planning:** After scaling the 5% sample back to full volume (÷ 0.05), the top hours represent hundreds of thousands of trips. Even marginal improvements in dispatch efficiency at peak hours translate to significant revenue and service gains.

### 4.1.2 Zone Positioning Recommendations

**Key Findings:**
- The **choropleth analysis** reveals that a handful of zones — particularly Midtown Manhattan, JFK Airport, LaGuardia Airport, and Penn Station — dominate pickup activity.
- The **pickup/dropoff ratio** analysis identifies zones that consistently generate more pickups than they receive dropoffs (high-demand source zones) and the reverse (sink zones where taxis become stranded).
- Night-hour analysis shows a distinct geographic shift toward entertainment districts and airports.

**Recommendations:**
1. **Rebalancing idle taxis:** Zones with a very low pickup/dropoff ratio (bottom 10 by ratio) are taxi graveyards — drivers drop passengers off but rarely find new fares there. Implement incentives or dispatch nudges to move idle taxis back to high-demand zones.
2. **Airport readiness:** JFK and LaGuardia consistently rank among the top pickup zones and generate high total_amount values (airport fee, long distances, tolls). Ensure adequate vehicle staging at both airports, especially during early-morning and late-evening flight windows.
3. **High-ratio zones as staging hubs:** Zones with the highest pickup/dropoff ratio are natural staging points — taxis that reposition there find fares quickly. Use these zones as preferred idle/waiting locations during off-peak hours.
4. **Borough-level deployment:** Manhattan accounts for the overwhelming majority of pickups. However, outer-borough zones with growing pickup counts (Brooklyn, Queens corridor) represent expansion opportunities, especially as ride-hailing competition intensifies in Manhattan.

### 4.1.3 Pricing Strategy Recommendations

**Key Findings:**
- **Fare per mile** is highest for short trips and decreases with distance, reflecting the NYC metered tariff structure (high base fare relative to per-mile rate for short distances).
- **Tip percentage** (credit card trips) is consistently higher for longer trips and varies by hour, with late-night trips attracting above-average tips.
- Surcharge application (congestion, airport fee, extra) varies significantly by zone and hour, directly impacting total revenue per trip.
- Vendor comparison shows minor but consistent differences in fare-per-mile across the day — suggesting opportunity for tariff optimisation.

**Recommendations:**
1. **Surge pricing windows:** The data supports targeted surge pricing during the 16:00–19:00 peak and late-night weekend hours, when demand significantly exceeds average. A modest 10–15% surge during these windows could meaningfully increase total revenue without suppressing demand, given the inelastic nature of taxi use in high-demand periods.
2. **Promote credit card payments:** Credit card payers tip at much higher rates than cash payers. Driver incentives and in-vehicle prompts to choose card payment could increase average per-trip revenue, particularly for longer trips where tip percentage has the most dollar impact.
3. **Short-trip premium justification:** Short trips (<2 miles) generate the highest fare per mile but the lowest tip percentage. Consider targeted short-trip incentives (e.g., reduced base fare) to increase volume in high-density zones where many short trips can be chained efficiently.
4. **Surcharge transparency:** Congestion and airport surcharges represent a significant and predictable revenue component. Ensuring drivers and dispatchers are aware of which zones and hours trigger these surcharges allows better trip selection to maximise total_amount per hour worked.
5. **Multi-passenger fare structure review:** Fare per mile per passenger decreases as group size increases — meaning larger groups are relatively cheap for the fleet to serve. Targeted marketing for group/shared rides in high-demand zones could increase vehicle occupancy and revenue per mile driven.